# HiveMind Agentic Benchmark: 4-Harness Arena (Core Suite & Polyglot Python)

Runtime: **A100 GPU** (40 GB oder 80 GB), High-RAM, >=100 GB Disk.

## Das Test-Setup (Harness vs. Harness auf identischem Modell)
Wir vergleichen nicht Modelle, sondern **Agenten-Harnesses**:
1. **`hivemind_duo`**: HiveMind Agentic Harness (Planner, Coder, Critic, AST-Guards, Auto-Lint, Fuzzy Diff, Multi-Turn Tool Loop)
2. **`aider`**: Aider CLI Harness (Diff Engine, Repo-Map)
3. **`hermes`**: Nous Research Hermes Agent CLI (`NousResearch/hermes-agent`)
4. **`pi`**: Mario Zechner Pi Coding Agent CLI (`earendil-works/pi`)

**Parität & Wissenschaftliche Sauberkeit:**
- **Identisches Modell & Gewichte:** Qwen3.6-35B-A3B (UD-Q4_K_XL) oder Hermes3.6-35B, voll auf die GPU geladen (76 GB VRAM-Budget, 0 MoE CPU Experts).
- **Gleiche Ressourcen:** Gleicher `llama-server`, identisches Modell, isolierte Workspaces, identisches Zeitlimit.
- **Kein Test-Leak:** Verdeckte Unit-Tests betreten niemals vor dem Check den Workspace. Volle Symmetrie.
- **Kein `raw`:** Rohe Zero-Shot-Diffs ohne Agenten-Tools messen nur das nackte Modell und verschwenden GPU-Stunden.

## Drei Phasen:
- **Phase 1: HiveMind Core Suite (23 Tasks, 32k Kontext)**: Multi-File, Refactorings, AST-Symbolerhalt, schwere Algorithmen (`hard_lru_cache`, `hard_cycle_detect`).
- **Phase 2: Polyglot Python Practice (34 Tasks, 32k Kontext)**: Der offizielle Python-Practice-Corpus aus `Aider-AI/polyglot-benchmark`.
- **Phase 3: DeepSWE Real-World Benchmark (Echte Repos, 128k Kontext)**: Reale Bugfixes aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`) mit 131.072 Kontext-Tokens.


## Ziele

| Ziel | Hardware | Modell / Kontext | Suites | Harnesses |
|---|---|---|---|---|
| **1. Small-GPU-Baseline** | T4 16 GB / L4 24 GB | `qwen3.5:4b-mtp`, 128k (nur wenn VRAM reicht) | 23 Core + 34 Polyglot, 3 Runs | HiveMind duo/full, Aider, Hermes, Pi |
| **2. Ceiling** | L4 24 GB / A100 40–80 GB | `qwen3.6:35b-a3b-ud`, 32k | 23 Core + 34 Polyglot | HiveMind duo/full, Aider, Hermes, Pi |
| **3. DeepSWE Real-World** | A100 40–80 GB | `qwen3.6:35b-a3b-ud`, 128k | 3 echte Repo-Tasks, 1 Run | HiveMind duo/full, Aider, Hermes, Pi |

**Wichtig:** 4B mit 128k ist auf einer T4 nicht garantiert; bei OOM zuerst 32k fahren. 35B mit 128k nur auf A100, nicht auf L4. DeepSWE ist ein 3-Task-Ausschnitt, nicht der volle 113er-Benchmark.


In [ ]:
# 1) GPU & Disk prüfen
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1


In [ ]:
# 2) Neuesten bench-stable Branch klonen / aktualisieren
![ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'


In [ ]:
# 3) Basis-Setup (~2-3 min: Python 3.14, llama.cpp, Aider, Hermes & Pi installieren)
# Installiert alle Tools und das Environment ohne langes Modell-Warten
!cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || true
!chmod +x /root/rent_bootstrap.sh
!HM_MODEL="skip" HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup


In [ ]:
# 3a) [DOWNLOAD OPTION A] 35B Modell herunterladen (~22 GB, ~10-15 min)
# Qwen 3.6 35B A3B UD-Q4_K_XL (MoE High-Performance Coder)
!bash /root/rent_bootstrap.sh download qwen3.6:35b-a3b-ud


In [ ]:
# 3b) [DOWNLOAD OPTION B] 4B Modell herunterladen (~3 GB, nur ~1-2 min)
# Qwen 3.5 4B MTP (Kompaktes 128k Modell - spart VRAM & Kosten, läuft auf jeder GPU)
!bash /root/rent_bootstrap.sh download qwen3.5:4b-mtp


In [ ]:
# 4a) [SERVER OPTION 1] 35B Modell in den VRAM laden & starten
# Qwen 3.6 35B MoE | Auto-VRAM Budget (L4: 2 MoE offloaded, A100: 100% VRAM)
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=32768 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


In [ ]:
# 4b) [SERVER OPTION 2] 4B Modell mit 128k Kontext in den VRAM laden & starten
# Qwen 3.5 4B MTP | Auto-Download falls fehlt (~2 Min) | T4: 128k machbar (~3.5GB), bei VRAM-Limit HM_CTX=32768
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.5:4b-mtp" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


In [ ]:
# 4c) VRAM- & Modell-Status prüfen
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
!echo "Aktives Modell:  $(cat /root/HiveMind_llama/.active_model 2>/dev/null || echo 'nicht gesetzt')"
!echo "Aktiver Kontext: $(cat /root/HiveMind_llama/.active_ctx 2>/dev/null || echo 'nicht gesetzt') Tokens"


In [ ]:
# 5) Smoke-Test (3 Tasks: HiveMind vs. Aider, ~5-10 min)
# Nutzt automatisch das aktive Modell aus Zelle 4a (35B) oder 4b (4B)
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh smoke


---
## PHASE 1: HiveMind Core Suite (23 Tasks)
Testet Multi-File-Navigation, Refactoring, AST-Symbolerhalt und Selbstkorrektur.
Head-to-Head: HiveMind Duo vs. Aider CLI auf identischem 35B-Modell.


In [ ]:
# 6a) Phase 1 starten: Core Suite (23 Tasks)
# Nutzt automatisch das Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell: 'hivemind_duo,aider'
#   - Volle 4-Harness Arena: 'hivemind_duo,aider,hermes,pi'
#   - Core vs. Full Toolset: 'hivemind_duo,hivemind_duo_full,aider,hermes,pi'
import os
os.environ['HM_RUNS'] = '3'  # 3 Durchläufe für statistische Varianz & Wilson CI (oder '1')
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 6b) Optional: Claude Code auf derselben Core Suite ausführen (braucht ANTHROPIC_API_KEY)
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')


In [ ]:
# 6c) Live-Status der Phase 1 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard (aktualisiert alle 10s; Interrupt stoppt nur die Anzeige):
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## PHASE 2: Polyglot Python Practice Benchmark (34 Tasks)
Das offizielle Benchmark-Set von Aider AI (`Aider-AI/polyglot-benchmark/python/exercises/practice`).
Testet den kompletten 34-Aufgaben-Corpus im direkten Head-to-Head auf demselben llama-server (~2-2.5h Laufzeit).


In [ ]:
# 7a) Alle 34 Polyglot-Python-Practice-Aufgaben von Aider AI herunterladen und importieren
!HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot


In [ ]:
# 7b) Phase 2 starten: Polyglot Python Practice (34 Tasks)
# Nutzt automatisch das Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell: 'hivemind_duo,aider'
#   - Volle 4-Harness Arena: 'hivemind_duo,aider,hermes,pi'
#   - Core vs. Full Toolset: 'hivemind_duo,hivemind_duo_full,aider,hermes,pi'
import os
os.environ['HM_RUNS'] = '3'  # 3 Durchläufe für statistische Varianz & Wilson CI (oder '1')
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_polyglot'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 7c) Optional: Claude Code auf dem vollen Aider-Benchmark ausführen
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks_polyglot
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')


In [ ]:
# 7d) Live-Status der Phase 2 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## PHASE 3: DeepSWE Real-World Benchmark (Echte Repositories, 128k Kontext)
Schwere, reale Software-Engineering-Aufgaben aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`).
Hier wird die volle Kontextstärke (128k / 131.072 Tokens) über alle Harnesses (`hivemind_duo`, `aider`, `hermes`, `pi`) getestet. Nur A100 + 35B; nicht auf L4/T4.
VRAM-Bedarf bei 128k: ~33 GB VRAM (passt perfekt in die Colab A100 mit 40/80 GB).


In [ ]:
# 8a) DeepSWE Python-Tasks importieren (lädt Repos & isolierte Hidden Tests ohne Docker)
# Installiert auch Test-Abhängigkeiten wie hypothesis und trio für den Test-Runner
!pip install -q hypothesis trio attrs pytest pytest-asyncio anyio
!bash /root/rent_bootstrap.sh deepswe


In [ ]:
# 8b) Engine mit 128k Kontext neu starten (~1 min)
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


In [ ]:
# 8c) Phase 3 starten: DeepSWE Real-World Tasks (128k Kontext)
# Nutzt automatisch das Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell: 'hivemind_duo,aider'
#   - Volle 4-Harness Arena: 'hivemind_duo,aider,hermes,pi'
#   - Core vs. Full Toolset: 'hivemind_duo,hivemind_duo_full,aider,hermes,pi'
import os
os.environ['HM_RUNS'] = '1'  # Bei DeepSWE (1800s Timeout) empfiehlt sich 1 Durchlauf
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_deepswe'
os.environ['HM_CTX'] = '131072'
os.environ['HM_CELL_TIMEOUT'] = '1800'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 8d) Live-Status der Phase 3 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py


---
## AUSWERTUNG & REPORT (Das vereinte Leaderboard)
Analysiert die Ergebnisse wissenschaftlich (Wilson 95% CI, McNemar-Test, Net-Token-Ratio vs. Aider).


In [ ]:
# 9) Unified Leaderboard generieren
# 9a) Getrennt nach Modell & Harness (z. B. hivemind_duo@35b vs. aider@35b vs. hivemind_duo@4b):
!cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py --by-model --results $(ls -1 /root/HiveMind_llama/bench/results/*.json 2>/dev/null)

# 9b) Gepoolt über alle Runs hinweg (mit Wilson 95% CI und McNemar-Test):
# !cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py --results $(ls -1 /root/HiveMind_llama/bench/results/*.json 2>/dev/null)


In [ ]:
# 10) Alle JSON-Ergebnisse dauerhaft in Google Drive sichern
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hivemind_bench
!cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
!cp -v /root/HiveMind_llama/bench/results/external/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
!cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
print('Ergebnisse erfolgreich in Google Drive gesichert!')


In [ ]:
# 11) Server stoppen
!bash /root/rent_bootstrap.sh stop
